# CAG-IDS: Context-Aware Graph Intrusion DetectionRun the cells in order. Each one prints what it did.Before you start: **Runtime -> Change runtime type -> T4 GPU -> Save**.

## 1. Mount Google Drive

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import osROOT = '/content/drive/MyDrive/cag-ids'for sub in ['data', 'code', 'outputs']:    os.makedirs(f'{ROOT}/{sub}', exist_ok=True)print(os.listdir(ROOT))

## 2. Get the code into DrivePick ONE of the two cells below.**Option A** if you pushed the code to GitHub (recommended, lets you pull updates).**Option B** if you uploaded the `code` folder to Drive by hand.

In [ ]:
# Option A: clone or pull from GitHubGITHUB_URL = 'https://github.com/YOURNAME/cag-ids.git'import os, subprocesstarget = f'{ROOT}/code'if os.path.exists(f'{target}/.git'):    print(subprocess.run(['git', '-C', target, 'pull'], capture_output=True, text=True).stdout)else:    import shutil    if os.path.exists(target):        shutil.rmtree(target)    print(subprocess.run(['git', 'clone', GITHUB_URL, target], capture_output=True, text=True).stdout)print(os.listdir(target))

In [ ]:
# Option B: you already uploaded the code folder to Drive. Just check it is there.import osexpected = ['configs', 'scripts', 'src', 'requirements.txt']present = os.listdir(f'{ROOT}/code')print('found:', present)missing = [e for e in expected if e not in present]print('missing:', missing if missing else 'nothing')

## 3. Install dependencies

In [ ]:
!pip install -q xgboost pyyaml pyarrowimport torchprint('torch', torch.__version__)print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable the T4 in Runtime settings')

## 4. Download the datasetNeeds a Kaggle API token: kaggle.com -> avatar -> Settings -> API -> Create New Token.Run the next cell and upload the `kaggle.json` file it asks for.Skip this section if the CSV is already in `MyDrive/cag-ids/data/`.

In [ ]:
import oscsv_path = f'{ROOT}/data/NF-UNSW-NB15-v3.csv'if os.path.exists(csv_path):    print('already downloaded: {:.2f} GB'.format(os.path.getsize(csv_path) / 1e9))else:    from google.colab import files    files.upload()

In [ ]:
import os, shutil, globcsv_path = f'{ROOT}/data/NF-UNSW-NB15-v3.csv'if not os.path.exists(csv_path):    for d in ['/root/.kaggle', '/root/.config/kaggle']:        os.makedirs(d, exist_ok=True)        shutil.copy('kaggle.json', f'{d}/kaggle.json')        os.chmod(f'{d}/kaggle.json', 0o600)    !pip install -q kaggle    !kaggle datasets download -d athena21/netflow-v3-datasets -f NF-UNSW-NB15-v3.csv -p /content    zips = glob.glob('/content/NF-UNSW-NB15-v3.csv*.zip')    if zips:        !unzip -o {zips[0]} -d /content    shutil.copy('/content/NF-UNSW-NB15-v3.csv', csv_path)print('{:.2f} GB'.format(os.path.getsize(csv_path) / 1e9))

## 5. Inspect the datasetRead this output carefully. It tells you three things:1. whether every column was recognised2. the class balance3. whether any attack class is missing from the test splitIf a class is missing from the test split, add `--split blocked` to thebuild_graphs command in section 7.

In [ ]:
CODE = f'{ROOT}/code'CFG = f'{CODE}/configs/default.yaml'!python {CODE}/scripts/explore_data.py --config {CFG} --root {ROOT} --nrows 500000

## 6. Prepare the dataThis cleans the flows, drops leaky and identifier columns, sorts by timeand assigns 60-second windows.Start with `--nrows 500000` to check the whole pipeline runs in a few minutes.Once that works, rerun without `--nrows` for the full 2.37M flows.

In [ ]:
!python {CODE}/scripts/prepare_data.py --config {CFG} --root {ROOT} --nrows 500000

## 7. Build the graphs and the host contextThis is the slow stage. It walks the windows in time order and computesstructural, behavioural and role context for every host, using onlycurrent and past windows.Add `--split blocked` here if section 5 warned about missing classes.

In [ ]:
!python {CODE}/scripts/build_graphs.py --config {CFG} --root {ROOT}

## 8. BaselinesRandom Forest, XGBoost and an MLP on flat flow features, no graph.If any of these scores above 0.99 macro F1, stop and investigate: that isalmost always a leaky feature, not a good model.

In [ ]:
!python {CODE}/scripts/run_baselines.py --config {CFG} --root {ROOT}

## 9. Train the graph modelsFour runs, in increasing order of what they know about each host:| experiment | node initialisation | fusion ||---|---|---|| `egraphsage` | vector of ones | none || `centrality_egraphsage` | structural centrality | none || `cagids_concat` | full context | concatenation || `cagids` | full context | learned gate |The last two differ only in how topology and context are combined, whichis what hypothesis H3 tests.

In [ ]:
!python {CODE}/scripts/train_gnn.py --config {CFG} --root {ROOT} --experiment egraphsage

In [ ]:
!python {CODE}/scripts/train_gnn.py --config {CFG} --root {ROOT} --experiment centrality_egraphsage

In [ ]:
!python {CODE}/scripts/train_gnn.py --config {CFG} --root {ROOT} --experiment cagids_concat

In [ ]:
!python {CODE}/scripts/train_gnn.py --config {CFG} --root {ROOT} --experiment cagids

## 10. AblationsEach one rebuilds the context with one group removed, then retrains.This is what answers "which context actually mattered".Run these only after section 9 has given you a working CAG-IDS number tocompare against.

In [ ]:
for tag, flag in [('no_structural', '--no-structural'),                  ('no_behavioural', '--no-behavioural'),                  ('no_role', '--no-role')]:    print('=' * 70)    print('ablation:', tag)    print('=' * 70)    !python {CODE}/scripts/build_graphs.py --config {CFG} --root {ROOT} {flag} --tag graphs_{tag}    !python {CODE}/scripts/train_gnn.py --config {CFG} --root {ROOT} --experiment cagids --graphs graphs_{tag} --tag {tag}

## 11. Window length sensitivityTests whether 60 seconds was a lucky choice.

In [ ]:
for seconds in [30, 300]:    print('=' * 70)    print('window:', seconds, 'seconds')    print('=' * 70)    !python {CODE}/scripts/build_graphs.py --config {CFG} --root {ROOT} --window-seconds {seconds} --tag graphs_w{seconds}    !python {CODE}/scripts/train_gnn.py --config {CFG} --root {ROOT} --experiment cagids --graphs graphs_w{seconds} --tag w{seconds}

## 12. Multiple seedsRun this for the numbers that go in the paper. Five seeds gives you themean and standard deviation the reviewers will expect.

In [ ]:
!python {CODE}/scripts/train_gnn.py --config {CFG} --root {ROOT} --experiment cagids --seeds 1 2 3 4 5 --tag cagids_5seeds

## 13. Final results tablePrints the comparison table and writes a LaTeX version you can pastestraight into the paper.

In [ ]:
!python {CODE}/scripts/summarize.py --config {CFG} --root {ROOT} --latex